In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
%pip install --no-build-isolation git+https://github.com/intel/auto-round.git --break-system-packages

  Cloning https://github.com/intel/auto-round.git to /tmp/pip-req-build-hulc6v0i
  Running command git clone --filter=blob:none --quiet https://github.com/intel/auto-round.git /tmp/pip-req-build-hulc6v0i
  Resolved https://github.com/intel/auto-round.git to commit a85997e19d97762172155c93c4dfc3fcc3976e71
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# %pip install git+https://gith/ub.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [21]:
import os

import torch
from auto_round import AutoRound, SignRoundConfig
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [24]:
import auto_round

print(auto_round.__version__)

0.17.0


In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    gpu_count = torch.cuda.device_count()
    print(f"Total GPUs Detected: {gpu_count}\n")

    for idx in range(gpu_count):
        props = torch.cuda.get_device_properties(idx)
        print(f"--- GPU {idx} ---")
        print(f"  Name: {props.name}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print(f"  Total VRAM: {props.total_memory / (1024**3):.2f} GB")
        print(f"  Allocated VRAM: {torch.cuda.memory_allocated(idx) / (1024**3):.2f} GB")
        print(f"  Free VRAM: {(props.total_memory - torch.cuda.memory_reserved(idx)) / (1024**3):.2f} GB\n")
else:
    print("No CUDA-compatible GPUs found.")

PyTorch Version: 2.15.0+cu132
CUDA Available: True
CUDA Version: 13.2
Total GPUs Detected: 1

--- GPU 0 ---
  Name: NVIDIA B200
  Compute Capability: 10.0
  Total VRAM: 178.34 GB
  Allocated VRAM: 0.00 GB
  Free VRAM: 178.34 GB



In [4]:
notebook_login()

In [5]:
MODEL_ID = "ukisai/Swift-1.5-Ornith-9B"
HF_USER = "Vishva007"               # Replace with your Hugging Face username
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 20 files:   0%|                                | 0/20 [00:00<?, ?it/s]

Fetching 20 files: 100%|██████████████████████| 20/20 [00:00<00:00, 130.40it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

In [34]:
from auto_round.utils.dataset_utils import CalibDataset

calib_dataset = [
    CalibDataset(
        "openbmb/UltraData-SFT-Agent-2609",
        num=205,
        apply_chat_template=True,
    ),
    CalibDataset(
        "nvidia/OpenCodeInstruct",
        num=154,
    ),
    CalibDataset(
        "qwedsacf/competition_math",
        num=102,
        concat=True,
    ),
    CalibDataset(
        "fineweb-edu",
        num=51,
    ),
]

In [35]:
# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1200,                                 # Optimization iterations per block; high accuracy (Production grade)
    gradient_accumulate_steps=4,                # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop    
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                                # Target pretrained model to be quantized
    tokenizer=tokenizer,                        # Tokenizer used to process text calibration samples
    processor=processor,                        # Processor for multimodal models (vision/audio), if present
    scheme="W4A16",                             # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,                     # Applies the algorithm-specific optimization settings defined above
    group_size=32,                              # Number of weights sharing one scale. Cannot go below 32 (vLLM's GPTQ kernel won't run it). Only use 16 if targeting NVFP4.
    sym=True,                                   # Symmetric quantization; enables faster compute and broader engine compatibility 
    nsamples=512,                               # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=2,                               # Batch size for calibration; keep < 8 to avoid slow execution/high memory overhead
    seqlen=4096,                                # Calibration context length; captures long-range dependency activations
    low_gpu_mem_usage=False,                    # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=True,                  # Compiles forward passes using torch.compile to accelerate calibration (JIT acceleration)
    quant_nontext_module=False,                 # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision (Crucial for VLM accuracy)
    dataset=calib_dataset,                      
    ignore_layers="linear_attn,mtp,visual,lm_head",     # Excludes linear-attention, MTP, and visual modules from quantization
    layer_config={
        "mtp": {
            "data_type": "bfloat16"
            },              # Keeps the MTP module in BF16 precision
        "mtp.fc": {
            "data_type": "bfloat16"
            },              # Keeps the MTP fully connected layer in BF16 precision
        "lm_head": {
            "data_type": "bfloat16"
            },              # Keeps head in BF16; Use at least 6-bit quantization, or at least 4-bit {"bits": 6} or {"bits": 4} if resources are limited.
    },                      # Preserves BF16 precision for the specified MTP layers
)

In [36]:
# Quantize once and export to disk across all major loader schemas
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # Serializes weights to 3 formats simultaneously to guarantee compatibility:
    # - 'auto_round': Reference AutoRound format (vLLM GPTQ/Marlin, Transformers / native loader)
    # - 'auto_gptq': AutoGPTQ tensor layout (vLLM GPTQ/Marlin, TGI)
    # - 'llm_compressor': Compressed-tensors schema (vLLM / Neural Magic)
    format="auto_round,auto_gptq,llm_compressor",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True,
)

2026-11-10 11:46:12 INFO layer_config_resolver.py L435: Ignored layers: linear_attn, lm_head, lm_head, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_a, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_b, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_qkv, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_z, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.out_proj, model.visual.blocks.[0-26].attn.proj, model.visual.blocks.[0-26].attn.qkv, model.visual.blocks.[0-26].mlp.linear_fc1, model.visual.blocks.[0-26].mlp.linear_fc2, model.visual.merger.linear_fc1, model.visual.merger.linear_fc2, mtp, visual
2026-11-10 11:46:12 INFO layer_config_resolver.py L435: Ignored layers: linear_attn, lm_head, lm_head, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].lin

2026-11-10 11:46:12 INFO base.py L1416: `torch.compile` is enabled
2026-11-10 11:46:13 INFO orchestrator.py L888: start to cache block inputs
2026-11-10 11:46:13 INFO mllm.py L88: Using MLLM template: qwen3_5
2026-11-10 11:46:13 INFO mllm.py L131: Multimodal model with non-MLLM calibration dataset 'openbmb/UltraData-SFT-Agent-2609:num=205:apply_chat_template=true,nvidia/OpenCodeInstruct:num=154,qwedsacf/competition_math:num=102:concat=true,fineweb-edu:num=51' and quant_nontext_module=False: using the standard text dataloader (vision/audio towers are not being quantized, so text-only calibration through the full-model forward is sufficient).
2026-11-10 11:46:13 INFO calib_dataset.py L2188: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
Process ForkProcess-7:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

TypeError: Couldn't cast array of type
struct<type: string, function: struct<name: string, description: string, parameters: struct<type: string, properties: string, required: list<item: string>>>, positional: list<item: string>>
to
{'type': Value('string'), 'function': {'name': Value('string'), 'description': Value('string'), 'parameters': {'type': Value('string'), 'properties': Json(decode=True), 'required': List(Value('string'))}}}

In [ ]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            delete_patterns=[
                "*.safetensors",
                "*.bin",
                "*.pt",
                "*.gguf",
                "*.json",
                "*.model",
                "*.onnx",
            ],
            commit_message="Update model weights and config",
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [ ]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [ ]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"),
        f"{base_name}-W4A16-AutoRound",
        hf_token,
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-gptq"),
        f"{base_name}-W4A16-GPTQ",
        hf_token,
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/llm-compressor-wint-a16"),
        f"{base_name}-W4A16-LLM-Compressor",
        hf_token,
    )

else:
    print("No Hugging Face token found. Skipping upload to hub.")